In [16]:
''' ML for FPL

    Learning projecting FPl points
    
    Sources:    f'{folder}in/Team_home.csv'
                f'{folder}in/Player_home.csv'
                f'{folder}mid/{source}/Team_xG.txt'
                f'{folder}mid/{source}/Team_xG_Ad.txt'
                f'{folder}mid/{source}/Team_Opponent_xG.txt'
                f'{folder}mid/{source}/Team_oppdefence_coeff.txt'
                
                folder in {'', '2019-2020'}
                source in {'FPL', 'Understat'}
                
    Writes:
                df

'''

#%pylab inline
import sklearn
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
from pathlib import Path
from Brr_functions import no_lists, del_empty_col
import constti
import json
# import imgkit

def x_maker( NAdf,  hist_num, NAdf_answers, NAdf_opp, NAdf_home, threatAllowedAv):
    '''
        Making table X for regression where last column ia an answer.
        Adjusted tables used for lag data. UnAdjusted for answer and opponent strength.
    '''
    #Removes empty fixtures
    df = NAlast(NAdf)
    df_answers =NAlast(NAdf_answers)
    df_opp = NAlast(NAdf_opp)
    df_home = NAlast(NAdf_home)
    
    X = pd.DataFrame()
    col = []

    for i in range(hist_num, 0, -1): #Creating column names for lag columns
        col.append(f'GW-{i}')

    for i in range(0,len(df.columns)-hist_num):
        #print(i+1)
        next_tour2add = df[df.columns[i:i+hist_num]].copy()#no_lists(df.sort_index())[df.columns[i:i+6]]
        next_tour2add.columns=col#['GW-6','GW-5','GW-4','GW-3','GW-2','GW-1']
        next_tour2add['Home'] = df_home[df_home.columns[hist_num+i]]
        next_tour2add['Opponent'] = (df_opp[df_opp.columns[hist_num+i]]*threatAllowedAv)#.copy()
        next_tour2add['GW'] = df_answers[df_answers.columns[hist_num+i]]#.copy()#.tolist()
        X = X.append(next_tour2add, ignore_index=True, sort=False)
    print(len(X.dropna()))
    return X.dropna()

def NAlast(df):
    '''
        Removes NaN holes in the table
    '''
    def NAlast_row(row):
        '''Removes holes in a row. Subfunction of NAlast.'''
        b = list(row.dropna())
        for i in range(len(b), len(row)):
            b.append(np.nan)
        return b
    dff = df.copy()
    for i in df.index:
        dff.loc[i] = NAlast_row(dff.loc[i])
    return dff

'''
    DOWNOADING TABLES FROM MID AND
    Making all input tables standart format. Only numBer columns. From left to right. No lists, columns for DGW added
'''

'''
    import curent year data
'''
# with open(Path('mid/FPL/Team_xG.txt'), 'r') as file:
#     Team_xG = pd.DataFrame(json.loads(file.read()))
#     Team_xG.index = pd.to_numeric(Team_xG.index)
#     TxG = no_lists(Team_xG.sort_index())[no_lists(Team_xG).columns[:3:-1]]
TxG = del_empty_col(pd.read_csv('mid/FPL/TxG.csv'))
TxGA = del_empty_col(pd.read_csv('mid/FPL/TxGA.csv'))
TOxG = del_empty_col(pd.read_csv('mid/FPL/TOxG.csv'))
    
# with open(Path('mid/FPL/Team_xG_Ad.txt'), 'r') as file:
#     Team_xG_Ad = pd.DataFrame(json.loads(file.read()))
#     Team_xG_Ad.index = pd.to_numeric(Team_xG_Ad.index)
#     TxGA = no_lists(Team_xG_Ad.sort_index())[TxG.columns]
#     #display(Team_xG_Ad)
    
# with open(Path('mid/FPL/Team_Opponent_xG.txt'), 'r') as file:
#     Team_Opponent_xG = pd.DataFrame(json.loads(file.read()))
#     Team_Opponent_xG.index = pd.to_numeric(Team_Opponent_xG.index)
#     TOxG = no_lists(Team_Opponent_xG.sort_index())[TxG.columns]
    
with open(Path('mid/FPL/Team_oppdefence_coeff.txt'), 'r') as file:
    Team_oppdefence_coeff = pd.DataFrame(json.loads(file.read()))
    Team_oppdefence_coeff.index = pd.to_numeric(Team_oppdefence_coeff.index)
    Todc = no_lists(Team_oppdefence_coeff.sort_index())[TxG.columns]
    
Team_home = pd.read_csv(Path(f'in/Team_home.csv'))
Player_home = pd.read_csv(Path(f'in/Player_home.csv'))

'''
    import 2019-2020 data
'''
year = '2019-2020'
if year=='': folder = ''
else: folder = f'history/{year}/'
with open(Path(f'{folder}mid/FPL/Team_xG.txt'), 'r') as file:
    Team_xG = pd.DataFrame(json.loads(file.read()))
    Team_xG.index = pd.to_numeric(Team_xG.index)
    TxG1920 = no_lists(Team_xG.sort_index())[no_lists(Team_xG).columns[:3:-1]]
    
with open(Path(f'{folder}mid/FPL/Team_xG_Ad.txt'), 'r') as file:
    Team_xG_Ad = pd.DataFrame(json.loads(file.read()))
    Team_xG_Ad.index = pd.to_numeric(Team_xG_Ad.index)
    TxGA1920 = no_lists(Team_xG_Ad.sort_index())[TxG1920.columns]
    #display(Team_xG_Ad)
    
with open(Path(f'{folder}mid/FPL/Team_Opponent_xG.txt'), 'r') as file:
    Team_Opponent_xG = pd.DataFrame(json.loads(file.read()))
    Team_Opponent_xG.index = pd.to_numeric(Team_Opponent_xG.index)
    TOxG1920 = no_lists(Team_Opponent_xG.sort_index())[TxG1920.columns]
    
with open(Path(f'{folder}mid/FPL/Team_oppdefence_coeff.txt'), 'r') as file:
    Team_oppdefence_coeff = pd.DataFrame(json.loads(file.read()))
    Team_oppdefence_coeff.index = pd.to_numeric(Team_oppdefence_coeff.index)
    Todc1920 = no_lists(Team_oppdefence_coeff.sort_index())[TxG1920.columns]
    
Team_home1920 = pd.read_csv(Path(f'{folder}in/Team_home.csv'))
Player_home1920 = pd.read_csv(Path(f'{folder}in/Player_home.csv'))

#Understat Data

'''
    import curent year data
'''
with open(Path('mid/Understat/Team_xG.txt'), 'r') as file:
    UnTeam_xG = pd.DataFrame(json.loads(file.read()))
    UnTeam_xG.index = pd.to_numeric(UnTeam_xG.index)
    UTxG = no_lists(UnTeam_xG.sort_index())[no_lists(UnTeam_xG).columns[:3:-1]]
    
with open(Path('mid/Understat/Team_xG_Ad.txt'), 'r') as file:
    UnTeam_xG_Ad = pd.DataFrame(json.loads(file.read()))
    UnTeam_xG_Ad.index = pd.to_numeric(UnTeam_xG_Ad.index)
    UTxGA = no_lists(UnTeam_xG_Ad.sort_index())[UTxG.columns]
    
with open(Path('mid/Understat/Team_Opponent_xG.txt'), 'r') as file:
    UnTeam_Opponent_xG = pd.DataFrame(json.loads(file.read()))
    UnTeam_Opponent_xG.index = pd.to_numeric(UnTeam_Opponent_xG.index)
    UTOxG = no_lists(UnTeam_Opponent_xG.sort_index())[TxG.columns]
    
with open(Path('mid/Understat/Team_oppdefence_coeff.txt'), 'r') as file:
    UnTeam_oppdefence_coeff = pd.DataFrame(json.loads(file.read()))
    UnTeam_oppdefence_coeff.index = pd.to_numeric(UnTeam_oppdefence_coeff.index)
    UTodc = no_lists(UnTeam_oppdefence_coeff.sort_index())[UTxG.columns]
    
'''
    import 2019-2020 data
'''
year = '2019-2020'
if year=='': folder = ''
else: folder = f'history/{year}/'
with open(Path(f'{folder}mid/Understat/Team_xG.txt'), 'r') as file:
    UnTeam_xG = pd.DataFrame(json.loads(file.read()))
    UnTeam_xG.index = pd.to_numeric(UnTeam_xG.index)
    UTxG1920 = no_lists(UnTeam_xG.sort_index())[no_lists(UnTeam_xG).columns[:3:-1]]
    
with open(Path(f'{folder}mid/Understat/Team_xG_Ad.txt'), 'r') as file:
    UnTeam_xG_Ad = pd.DataFrame(json.loads(file.read()))
    UnTeam_xG_Ad.index = pd.to_numeric(UnTeam_xG_Ad.index)
    UTxGA1920 = no_lists(UnTeam_xG_Ad.sort_index())[UTxG1920.columns]
    #display(Team_xG_Ad)
    
with open(Path(f'{folder}mid/Understat/Team_Opponent_xG.txt'), 'r') as file:
    UnTeam_Opponent_xG = pd.DataFrame(json.loads(file.read()))
    UnTeam_Opponent_xG.index = pd.to_numeric(UnTeam_Opponent_xG.index)
    UTOxG1920 = no_lists(UnTeam_Opponent_xG.sort_index())[UTxG1920.columns]
    
with open(Path(f'{folder}mid/Understat/Team_oppdefence_coeff.txt'), 'r') as file:
    UnTeam_oppdefence_coeff = pd.DataFrame(json.loads(file.read()))
    UnTeam_oppdefence_coeff.index = pd.to_numeric(UnTeam_oppdefence_coeff.index)
    UTodc1920 = no_lists(UnTeam_oppdefence_coeff.sort_index())[UTxG1920.columns]

display(UTOxG1920.mean(axis=1).mean())#That's threatAllowedAv actually


144.51298014769435

In [40]:
#def x_maker( NAdf,  hist_num, NAdf_answers, NAdf_opp, NAdf_home, threatAllowedAv)
hist_num = 6
threatAllowedAv = TOxG.mean(axis=1).mean()
#X3 = x_maker(TxGA1920,  hist_num, TxG1920, Todc1920, Team_home1920, TOxG1920.mean(axis=1).mean())
df = x_maker(TxGA,  hist_num, TxG, Todc, Team_home, threatAllowedAv)
#X = pd.DataFrame(df[df.columns[:-3]]) # if autoregression, not mean
X = pd.DataFrame(df[df.columns[:-3]].mean(axis=1))
X['Home'] = df['Home']
X['Opponent'] = df['Opponent']
#X = X3[X3.columns[:-1]]
Y = df['GW']
reg = LinearRegression().fit(X, Y)
display(reg.score(X, Y))
display(reg.coef_)

308


0.38305787805935787

array([ 0.6221257 , 20.03740264,  1.08137848])

In [30]:
TDw = pd.read_csv('mid/FPL/Team_Defence_weight.csv')
TAw = pd.read_csv('mid/FPL/Team_Attack_weight.csv')
Todc
#TDw[TDw.columns[:23]]/TOxG.mean(axis=1).mean()
TAw[TAw.columns[:23]]/TOxG.mean(axis=1).mean()

,GW1,GW2,GW3,GW4,GW5,GW6,GW7,GW8,GW9,GW10,...,GW14,GW15,GW16,GW17,GW18,GW19,GW19*,GW20,GW21,GW22
0,1.097791,0.859421,0.684290,1.137197,0.530973,0.876239,0.929880,1.099735,1.235698,0.927270,...,1.050936,0.705939,0.800561,1.445997,1.053980,1.187359,NaN,1.013345,0.929880,0.927270
1,NaN,1.137197,1.097791,0.684290,0.876239,1.235698,1.013345,0.880588,0.800561,0.859421,...,1.445997,1.053980,0.705939,0.929880,NaN,0.530973,1.187359,1.356595,1.013345,0.859421
2,0.705939,1.187359,0.929880,1.050936,1.053980,1.445997,1.126207,1.356595,1.099735,0.684290,...,1.137197,0.859421,0.880588,0.927270,0.530973,1.235698,NaN,1.097791,1.126207,0.684290
3,NaN,0.876239,1.013345,1.187359,1.445997,1.126207,0.705939,0.800561,1.053980,0.530973,...,0.927270,1.235698,1.137197,NaN,0.929880,0.859421,0.684290,1.099735,0.705939,0.530973
4,0.800561,0.684290,1.445997,1.053980,1.013345,0.929880,1.356595,1.137197,1.187359,1.126207,...,0.859421,0.880588,1.099735,0.530973,NaN,1.097791,0.876239,0.927270,1.356595,1.126207
5,1.013345,0.929880,1.050936,0.705939,0.800561,1.097791,0.927270,1.235698,1.356595,1.187359,...,0.684290,1.099735,0.876239,1.137197,0.880588,0.530973,NaN,0.859421,0.927270,1.187359
6,1.126207,1.445997,1.053980,0.800561,0.684290,1.013345,1.187359,0.929880,1.097791,1.235698,...,0.880588,1.137197,NaN,0.859421,0.927270,NaN,NaN,0.876239,1.187359,1.235698
7,0.880588,1.235698,1.099735,0.927270,1.137197,1.053980,1.445997,0.859421,1.050936,0.876239,...,1.187359,1.013345,NaN,NaN,1.126207,0.705939,0.929880,0.800561,1.445997,0.876239
8,1.445997,1.356595,0.530973,0.859421,1.099735,0.880588,1.235698,0.927270,0.684290,1.097791,...,1.126207,0.929880,1.053980,1.187359,NaN,1.013345,0.705939,1.050936,1.235698,1.097791
9,0.684290,1.097791,1.137197,0.530973,0.927270,1.099735,0.876239,1.053980,0.880588,1.050936,...,0.929880,1.356595,1.445997,1.126207,NaN,0.800561,NaN,1.187359,0.876239,1.050936


In [58]:
X2 = x_maker(TxGA1920[TxGA1920.columns[30:]],  hist_num, TxG1920[TxGA1920.columns[30:]],\
Todc1920[TxGA1920.columns[30:]], Team_home1920[TxGA1920.columns[30:]], TOxG1920[TxGA1920.columns[30:]].mean(axis=1).mean())
X2[X2.columns[:-3]].mean(axis=1)

180    142.000306
181    145.738690
190    255.457395
194    103.225981
200    155.634941
201    133.472682
210    271.458544
214    132.694862
220    141.099396
221    131.413699
222    154.753510
223    122.257216
224    116.816313
225    247.971866
226    150.413155
227    105.210873
228    173.712754
229    250.891365
230    251.317389
231    176.320088
232    124.799596
233     87.040689
234    137.497632
235    187.628561
236    133.285120
237    128.061204
238    184.251215
239    108.577514
240    128.089725
241    126.437848
242    171.704881
243    112.662173
244    137.306857
245    205.003064
246    169.877115
247     94.473306
248    203.160942
249    247.290722
250    284.519632
251    155.780153
252    134.117816
253     78.364710
254    149.745990
255    202.326802
256    135.810663
257    116.704811
258    198.048536
259    120.112082
dtype: float64

In [24]:
Table = pd.read_csv("in/Table_FPL.csv")
Table[Table['was_home']==False]['threat'].sum()/89

154.1573033707865